In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [5]:
%%writefile matmul.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <math.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <iostream>

__global__ void matmul(float *A,float *B,float *C,int M,int K,int N){
    int row=threadIdx.x+blockIdx.x*blockDim.x;
    int col=threadIdx.y+blockIdx.y*blockDim.y;

    if(row<M && col<N){
        float sumt=0.0f;
        for(int i=0;i<K;i++){
            sumt+=A[row*K+i]*B[i*N+col];
        }
        C[row*N+col]=sumt;
    }
}

using namespace std;

int main(int argc,char *argv[]){
    if (argc < 4) {
        printf("Usage: ./matmul_cublas M K N\n");
        return 1;
    }

    int M = atoi(argv[1]);
    int K = atoi(argv[2]);
    int N = atoi(argv[3]);

    dim3 threads(32,32);
    int tx=(M+32-1)/32;
    int ty=(N+32-1)/32;
    dim3 blocks(tx,ty);

    size_t sizeA=M*K*sizeof(float);
    size_t sizeB=N*K*sizeof(float);
    size_t sizeC=M*N*sizeof(float);
    float *A=(float*)malloc(sizeA);
    float *B=(float*)malloc(sizeB);
    float *C=(float*)malloc(sizeC);

    for(int i=0;i<M*K;i++){
        A[i]=1.0f;
    }
    for(int i=0;i<K*N;i++){
        B[i]=1.0f;
    }
    float *dA,*dB,*dC;
    cudaMalloc(&dA,sizeA);
    cudaMalloc(&dB,sizeB);
    cudaMalloc(&dC,sizeC);

    cudaEvent_t events[4];
    for(int i=0;i<4;i++) cudaEventCreate(&events[i]);

    cudaEventRecord(events[0]);
    cudaMemcpy(dA,A,sizeA,cudaMemcpyHostToDevice);
    cudaMemcpy(dB,B,sizeB,cudaMemcpyHostToDevice);

    cudaEventRecord(events[1]);
    matmul<<<blocks,threads>>>(dA,dB,dC,M,K,N);
    cudaEventRecord(events[2]);
    cudaDeviceSynchronize();

    cudaMemcpy(C,dC,sizeC,cudaMemcpyDeviceToHost);
    cudaEventRecord(events[3]);

    cudaEventSynchronize(events[2]);
    float mt=0.0f;
    cudaEventElapsedTime(&mt,events[1],events[2]);
    printf("kernel time: %.3f ms\n", mt);

    cudaEventSynchronize(events[3]);
    float mst=0.0f;
    cudaEventElapsedTime(&mst,events[0],events[3]);
    printf("overall time: %.3f ms\n", mst);

    for(int i=0;i<M*N;i++){
        if(C[i]-K>1e-5){
            cout<<"wrong";
            free(A);
            free(B);
            free(C);
            cudaFree(dA);
            cudaFree(dB);
            cudaFree(dC);
            exit(1);
        }
    }

    cout<<"Works Correctly";
    free(A);
    free(B);
    free(C);
    cudaFree(dA);
    cudaFree(dB);
    cudaFree(dC);
    return 0;
}

Overwriting matmul.cu


In [6]:
!nvcc matmul.cu -o matmul

In [7]:
!nvprof ./matmul 1024 1025 1025

==194== NVPROF is profiling process 194, command: ./matmul 1024 1025 1025
kernel time: 182.774 ms
overall time: 189.075 ms
Works Correctly==194== Profiling application: ./matmul 1024 1025 1025
==194== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   85.20%  25.391ms         1  25.391ms  25.391ms  25.391ms  matmul(float*, float*, float*, int, int, int)
                    9.59%  2.8568ms         1  2.8568ms  2.8568ms  2.8568ms  [CUDA memcpy DtoH]
                    5.21%  1.5534ms         2  776.70us  776.12us  777.27us  [CUDA memcpy HtoD]
      API calls:   51.47%  210.26ms         3  70.086ms  102.39us  210.04ms  cudaMalloc
                   38.54%  157.46ms         1  157.46ms  157.46ms  157.46ms  cudaLaunchKernel
                    6.22%  25.401ms         1  25.401ms  25.401ms  25.401ms  cudaDeviceSynchronize
                    1.51%  6.1772ms         3  2.0591ms  1.0020ms  3.8324ms  cudaMemcpy
               

In [8]:
%%writefile matmul2.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <math.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <iostream>

__global__ void matmul(float *A,float *B,float *C,int M,int K,int N){
    int col=threadIdx.x+blockIdx.x*blockDim.x;
    int row=threadIdx.y+blockIdx.y*blockDim.y;

    if(row<M && col<N){
        float sumt=0.0f;
        for(int i=0;i<K;i++){
            sumt+=A[row*K+i]*B[i*N+col];
        }
        C[row*N+col]=sumt;
    }
}

using namespace std;

int main(int argc,char *argv[]){
    if (argc < 4) {
        printf("Usage: ./matmul_cublas M K N\n");
        return 1;
    }

    int M = atoi(argv[1]);
    int K = atoi(argv[2]);
    int N = atoi(argv[3]);

    dim3 threads(32,32);
    int tx=(N+32-1)/32;
    int ty=(M+32-1)/32;
    dim3 blocks(tx,ty);

    size_t sizeA=M*K*sizeof(float);
    size_t sizeB=N*K*sizeof(float);
    size_t sizeC=M*N*sizeof(float);
    float *A=(float*)malloc(sizeA);
    float *B=(float*)malloc(sizeB);
    float *C=(float*)malloc(sizeC);

    for(int i=0;i<M*K;i++){
        A[i]=1.0f;
    }
    for(int i=0;i<K*N;i++){
        B[i]=1.0f;
    }
    float *dA,*dB,*dC;
    cudaMalloc(&dA,sizeA);
    cudaMalloc(&dB,sizeB);
    cudaMalloc(&dC,sizeC);

    cudaEvent_t events[4];
    for(int i=0;i<4;i++) cudaEventCreate(&events[i]);

    cudaEventRecord(events[0]);
    cudaMemcpy(dA,A,sizeA,cudaMemcpyHostToDevice);
    cudaMemcpy(dB,B,sizeB,cudaMemcpyHostToDevice);

    cudaEventRecord(events[1]);
    matmul<<<blocks,threads>>>(dA,dB,dC,M,K,N);
    cudaEventRecord(events[2]);
    cudaDeviceSynchronize();

    cudaMemcpy(C,dC,sizeC,cudaMemcpyDeviceToHost);
    cudaEventRecord(events[3]);

    cudaEventSynchronize(events[2]);
    float mt=0.0f;
    cudaEventElapsedTime(&mt,events[1],events[2]);
    printf("kernel time: %.3f ms\n", mt);

    cudaEventSynchronize(events[3]);
    float mst=0.0f;
    cudaEventElapsedTime(&mst,events[0],events[3]);
    printf("overall time: %.3f ms\n", mst);

    for(int i=0;i<M*N;i++){
        if(C[i]-K>1e-5){
            cout<<"wrong";
            free(A);
            free(B);
            free(C);
            cudaFree(dA);
            cudaFree(dB);
            cudaFree(dC);
            exit(1);
        }
    }

    cout<<"Works Correctly";
    free(A);
    free(B);
    free(C);
    cudaFree(dA);
    cudaFree(dB);
    cudaFree(dC);
    return 0;
}

Writing matmul2.cu


In [9]:
!nvcc matmul2.cu -o matmul2

In [10]:
!nvprof ./matmul2 1024 1025 1025

==263== NVPROF is profiling process 263, command: ./matmul2 1024 1025 1025
kernel time: 27.561 ms
overall time: 33.176 ms
Works Correctly==263== Profiling application: ./matmul2 1024 1025 1025
==263== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   55.71%  5.3182ms         1  5.3182ms  5.3182ms  5.3182ms  matmul(float*, float*, float*, int, int, int)
                   28.26%  2.6980ms         1  2.6980ms  2.6980ms  2.6980ms  [CUDA memcpy DtoH]
                   16.04%  1.5309ms         2  765.45us  758.23us  772.66us  [CUDA memcpy HtoD]
      API calls:   82.65%  186.11ms         3  62.036ms  100.25us  185.91ms  cudaMalloc
                    9.91%  22.317ms         1  22.317ms  22.317ms  22.317ms  cudaLaunchKernel
                    2.45%  5.5175ms         3  1.8392ms  954.18us  3.5591ms  cudaMemcpy
                    2.36%  5.3160ms         1  5.3160ms  5.3160ms  5.3160ms  cudaDeviceSynchronize
               

In [27]:
%%writefile matmul3.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <math.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <iostream>
#include <cublas_v2.h>

using namespace std;

int main(int argc,char *argv[]){
    if (argc < 4) {
        printf("Usage: ./matmul_cublas M K N\n");
        return 1;
    }

    int M = atoi(argv[1]);
    int K = atoi(argv[2]);
    int N = atoi(argv[3]);

    size_t sizeA=M*K*sizeof(float);
    size_t sizeB=N*K*sizeof(float);
    size_t sizeC=M*N*sizeof(float);
    float *A=(float*)malloc(sizeA);
    float *B=(float*)malloc(sizeB);
    float *C=(float*)malloc(sizeC);

    for(int i=0;i<M*K;i++){
        A[i]=1.0f;
    }
    for(int i=0;i<K*N;i++){
        B[i]=1.0f;
    }
    float *dA,*dB,*dC;
    cudaMalloc(&dA,sizeA);
    cudaMalloc(&dB,sizeB);
    cudaMalloc(&dC,sizeC);

    cublasHandle_t handle;
    cublasCreate(&handle);
    float alpha=1.0f;
    float beta=0.0f;

    cudaEvent_t events[4];
    for(int i=0;i<4;i++) cudaEventCreate(&events[i]);
    
    cudaEventRecord(events[0]);
    cudaMemcpy(dA,A,sizeA,cudaMemcpyHostToDevice);
    cudaMemcpy(dB,B,sizeB,cudaMemcpyHostToDevice);

    cudaEventRecord(events[1]);

    cublasSgemm(handle,CUBLAS_OP_N,CUBLAS_OP_N,M,N,K,&alpha,dB,K,dA,N,&beta,dC,N);
    
    cudaEventRecord(events[2]);
    cudaDeviceSynchronize();

    cudaMemcpy(C,dC,sizeC,cudaMemcpyDeviceToHost);
    cudaEventRecord(events[3]);

    cudaEventSynchronize(events[2]);
    float mt=0.0f;
    cudaEventElapsedTime(&mt,events[1],events[2]);
    printf("kernel time: %.3f ms\n", mt);

    cudaEventSynchronize(events[3]);
    float mst=0.0f;
    cudaEventElapsedTime(&mst,events[0],events[3]);
    printf("overall time: %.3f ms\n", mst);

    cublasDestroy(handle);
    
    for(int i=0;i<M*N;i++){
        if(C[i]-K>1e-5){
            cout<<"wrong";
            free(A);
            free(B);
            free(C);
            cudaFree(dA);
            cudaFree(dB);
            cudaFree(dC);
            exit(1);
        }
    }

    cout<<"Works Correctly";
    free(A);
    free(B);
    free(C);
    cudaFree(dA);
    cudaFree(dB);
    cudaFree(dC);
    return 0;
}

Overwriting matmul3.cu


In [29]:
!nvcc matmul3.cu -lcublas -o matmul3

In [30]:
!nvprof ./matmul3 1024 1025 1025

==722== NVPROF is profiling process 722, command: ./matmul3 1024 1025 1025
kernel time: 82.668 ms
overall time: 88.254 ms
Works Correctly==722== Profiling application: ./matmul3 1024 1025 1025
==722== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   54.00%  2.5964ms         1  2.5964ms  2.5964ms  2.5964ms  [CUDA memcpy DtoH]
                   33.41%  1.6065ms         2  803.27us  794.30us  812.25us  [CUDA memcpy HtoD]
                   12.58%  604.76us         1  604.76us  604.76us  604.76us  volta_sgemm_128x64_nn
                    0.02%     736ns         1     736ns     736ns     736ns  [CUDA memset]
      API calls:   61.04%  188.64ms         6  31.439ms  5.8990us  186.78ms  cudaMalloc
                   18.88%  58.360ms         1  58.360ms  58.360ms  58.360ms  cudaGetSymbolAddress
                   10.38%  32.069ms         2  16.035ms  1.2200us  32.068ms  cudaOccupancyMaxActiveBlocksPerMultiprocessor
        

In [4]:
%%writefile tiling.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <math.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <iostream>

#define TILE 32

__global__ void matmul(float *A,float *B,float *C,int M,int K,int N){
    __shared__ float sA[TILE*TILE];
    __shared__ float sB[TILE*TILE];
    
    int globalrow=threadIdx.y+blockIdx.y*blockDim.y;
    int globalcol=threadIdx.x+blockIdx.x*blockDim.x;
    int localrow=threadIdx.y;
    int localcol=threadIdx.x;

    float acc=0.0f;
    int  numtiles=(K+TILE-1)/TILE;

    for(int i=0;i<numtiles;i++){
        int Arow=globalrow;
        int Acol=i*TILE+localcol;
        int Brow=i*TILE+localrow;
        int Bcol=globalcol;

        if(Arow<M && Acol<K){
            sA[localrow*TILE + localcol]=A[Arow*K + Acol];
        }
        else{
            sA[localrow*TILE + localcol]=0.0f;
        }

        if(Brow<K && Bcol<N){
            sB[localrow*TILE + localcol]=B[Brow*N + Bcol];
        }
        else{
            sB[localrow*TILE + localcol]=0.0f;
        }

        __syncthreads();

        for(int k=0;k<TILE;k++){
            acc+=sA[localrow*TILE+k]*sB[k*TILE+localcol];
        }

        __syncthreads();
    }

    if(globalrow<M && globalcol<N){
        C[globalrow*N+globalcol]=acc;
    }
    
}

using namespace std;

int main(int argc,char *argv[]){
    if (argc < 4) {
        printf("Usage: ./matmul_cublas M K N\n");
        return 1;
    }

    int M = atoi(argv[1]);
    int K = atoi(argv[2]);
    int N = atoi(argv[3]);

    dim3 threads(32,32);
    int tx=(N+TILE-1)/TILE;
    int ty=(M+TILE-1)/TILE;
    dim3 blocks(tx,ty);

    size_t sizeA=M*K*sizeof(float);
    size_t sizeB=N*K*sizeof(float);
    size_t sizeC=M*N*sizeof(float);
    float *A=(float*)malloc(sizeA);
    float *B=(float*)malloc(sizeB);
    float *C=(float*)malloc(sizeC);

    for(int i=0;i<M*K;i++){
        A[i]=1.0f;
    }
    for(int i=0;i<K*N;i++){
        B[i]=1.0f;
    }
    float *dA,*dB,*dC;
    cudaMalloc(&dA,sizeA);
    cudaMalloc(&dB,sizeB);
    cudaMalloc(&dC,sizeC);

    cudaEvent_t events[4];
    for(int i=0;i<4;i++) cudaEventCreate(&events[i]);

    cudaEventRecord(events[0]);
    cudaMemcpy(dA,A,sizeA,cudaMemcpyHostToDevice);
    cudaMemcpy(dB,B,sizeB,cudaMemcpyHostToDevice);

    cudaEventRecord(events[1]);
    matmul<<<blocks,threads>>>(dA,dB,dC,M,K,N);
    cudaEventRecord(events[2]);
    cudaDeviceSynchronize();

    cudaMemcpy(C,dC,sizeC,cudaMemcpyDeviceToHost);
    cudaEventRecord(events[3]);

    cudaEventSynchronize(events[2]);
    float mt=0.0f;
    cudaEventElapsedTime(&mt,events[1],events[2]);
    printf("kernel time: %.3f ms\n", mt);

    cudaEventSynchronize(events[3]);
    float mst=0.0f;
    cudaEventElapsedTime(&mst,events[0],events[3]);
    printf("overall time: %.3f ms\n", mst);

    for(int i=0;i<M*N;i++){
        if(fabs(C[i]-K)>1e-5){
            cout<<"wrong";
            free(A);
            free(B);
            free(C);
            cudaFree(dA);
            cudaFree(dB);
            cudaFree(dC);
            exit(1);
        }
    }

    cout<<"Works Correctly";
    free(A);
    free(B);
    free(C);
    cudaFree(dA);
    cudaFree(dB);
    cudaFree(dC);
    return 0;
}

Overwriting tiling.cu


In [5]:
!nvcc tiling.cu -o tiling

In [9]:
!nvprof ./tiling 1024 1025 1025

==193== NVPROF is profiling process 193, command: ./tiling 1024 1025 1025
kernel time: 5.856 ms
overall time: 11.581 ms
Works Correctly==193== Profiling application: ./tiling 1024 1025 1025
==193== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   57.40%  5.7053ms         1  5.7053ms  5.7053ms  5.7053ms  matmul(float*, float*, float*, int, int, int)
                   26.96%  2.6803ms         1  2.6803ms  2.6803ms  2.6803ms  [CUDA memcpy DtoH]
                   15.64%  1.5547ms         2  777.37us  771.32us  783.42us  [CUDA memcpy HtoD]
      API calls:   91.44%  182.89ms         3  60.965ms  101.14us  182.68ms  cudaMalloc
                    2.85%  5.7083ms         1  5.7083ms  5.7083ms  5.7083ms  cudaDeviceSynchronize
                    2.81%  5.6191ms         3  1.8730ms  947.10us  3.6828ms  cudaMemcpy
                    2.36%  4.7236ms       228  20.717us     106ns  1.3230ms  cuDeviceGetAttribute
              

In [13]:
%%writefile tiling2.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <math.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <iostream>

#define TILE 32
#define NELEM 4

__global__ void matmul(float *A,float *B,float *C,int M,int K,int N){
    __shared__ float sA[TILE*TILE];
    __shared__ float sB[TILE*TILE*NELEM];
    
    int globalrow=threadIdx.y+blockIdx.y*blockDim.y;
    int globalcol=threadIdx.x+blockIdx.x*blockDim.x*NELEM;
    int localrow=threadIdx.y;
    int localcol=threadIdx.x;

    float acc[NELEM]={0};
    int  numtiles=(K+TILE-1)/TILE;
    int sizetile=TILE*TILE;
    
    for(int i=0;i<numtiles;i++){
        int Arow=globalrow;
        int Acol=i*TILE+localcol;
        

        if(Arow<M && Acol<K){
            sA[localrow*TILE + localcol]=A[Arow*K + Acol];
        }
        else{
            sA[localrow*TILE + localcol]=0.0f;
        }

        for(int y=0;y<NELEM;y++){
            int Brow=i*TILE+localrow;
            int Bcol=globalcol+y*TILE;
            if(Brow<K && Bcol<N){
                sB[sizetile*y+localrow*TILE + localcol]=B[Brow*N + Bcol];
            }
            else{
                sB[sizetile*y+localrow*TILE + localcol]=0.0f;
            }
        }
        
        __syncthreads();

        for(int k=0;k<TILE;k++){
            float a=sA[localrow*TILE+k];
            for(int y=0;y<NELEM;y++){
                acc[y]+=a*sB[k*TILE+localcol + y*sizetile];
            }
        }

        __syncthreads();
    }

    for(int y=0;y<NELEM;y++){
        int gcol=globalcol+y*TILE;
        if(globalrow<M && gcol<N){
            C[globalrow*N+gcol]=acc[y];
        }
    }
}

using namespace std;

int main(int argc,char *argv[]){
    if (argc < 4) {
        printf("Usage: ./matmul_cublas M K N\n");
        return 1;
    }

    int M = atoi(argv[1]);
    int K = atoi(argv[2]);
    int N = atoi(argv[3]);

    dim3 threads(32,32);
    int tx=(N+NELEM*TILE-1)/(NELEM*TILE);
    int ty=(M+TILE-1)/TILE;
    dim3 blocks(tx,ty);

    size_t sizeA=M*K*sizeof(float);
    size_t sizeB=N*K*sizeof(float);
    size_t sizeC=M*N*sizeof(float);
    float *A=(float*)malloc(sizeA);
    float *B=(float*)malloc(sizeB);
    float *C=(float*)malloc(sizeC);

    for(int i=0;i<M*K;i++){
        A[i]=1.0f;
    }
    for(int i=0;i<K*N;i++){
        B[i]=1.0f;
    }
    float *dA,*dB,*dC;
    cudaMalloc(&dA,sizeA);
    cudaMalloc(&dB,sizeB);
    cudaMalloc(&dC,sizeC);

    cudaEvent_t events[4];
    for(int i=0;i<4;i++) cudaEventCreate(&events[i]);

    cudaEventRecord(events[0]);
    cudaMemcpy(dA,A,sizeA,cudaMemcpyHostToDevice);
    cudaMemcpy(dB,B,sizeB,cudaMemcpyHostToDevice);

    cudaEventRecord(events[1]);
    matmul<<<blocks,threads>>>(dA,dB,dC,M,K,N);
    cudaEventRecord(events[2]);
    cudaDeviceSynchronize();

    cudaMemcpy(C,dC,sizeC,cudaMemcpyDeviceToHost);
    cudaEventRecord(events[3]);

    cudaEventSynchronize(events[2]);
    float mt=0.0f;
    cudaEventElapsedTime(&mt,events[1],events[2]);
    printf("kernel time: %.3f ms\n", mt);

    cudaEventSynchronize(events[3]);
    float mst=0.0f;
    cudaEventElapsedTime(&mst,events[0],events[3]);
    printf("overall time: %.3f ms\n", mst);

    for(int i=0;i<M*N;i++){
        if(fabs(C[i]-K)>1e-5){
            cout<<"wrong";
            free(A);
            free(B);
            free(C);
            cudaFree(dA);
            cudaFree(dB);
            cudaFree(dC);
            exit(1);
        }
    }

    cout<<"Works Correctly";
    free(A);
    free(B);
    free(C);
    cudaFree(dA);
    cudaFree(dB);
    cudaFree(dC);
    return 0;
}

Overwriting tiling2.cu


In [14]:
!nvcc tiling2.cu -o tiling2

In [15]:
!nvprof ./tiling2 1024 1025 1025

==245== NVPROF is profiling process 245, command: ./tiling2 1024 1025 1025
kernel time: 29.607 ms
overall time: 35.076 ms
Works Correctly==245== Profiling application: ./tiling2 1024 1025 1025
==245== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   52.90%  4.6196ms         1  4.6196ms  4.6196ms  4.6196ms  matmul(float*, float*, float*, int, int, int)
                   29.70%  2.5941ms         1  2.5941ms  2.5941ms  2.5941ms  [CUDA memcpy DtoH]
                   17.40%  1.5192ms         2  759.61us  758.33us  760.89us  [CUDA memcpy HtoD]
      API calls:   81.94%  185.05ms         3  61.685ms  96.907us  184.85ms  cudaMalloc
                   11.10%  25.061ms         1  25.061ms  25.061ms  25.061ms  cudaLaunchKernel
                    2.37%  5.3613ms         3  1.7871ms  946.31us  3.4677ms  cudaMemcpy
                    2.17%  4.9025ms       228  21.502us     106ns  1.4332ms  cuDeviceGetAttribute
                

In [16]:
%%writefile tiling3.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <math.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <iostream>

#define TILE 32
#define NELEM 4

__global__ void matmul(float *A,float *B,float *C,int M,int K,int N){
    __shared__ float sA[TILE*TILE*NELEM];
    __shared__ float sB[TILE*TILE];
    
    int globalrow=threadIdx.y+blockIdx.y*blockDim.y*NELEM;
    int globalcol=threadIdx.x+blockIdx.x*blockDim.x;
    int localrow=threadIdx.y;
    int localcol=threadIdx.x;

    float acc[NELEM]={0};
    int  numtiles=(K+TILE-1)/TILE;
    int sizetile=TILE*TILE;
    
    for(int i=0;i<numtiles;i++){
        int Brow=i*TILE+localrow;
        int Bcol=globalcol;
        if(Brow<K && Bcol<N){
            sB[localrow*TILE + localcol]=B[Brow*N + Bcol];
        }
        else{
            sB[localrow*TILE + localcol]=0.0f;
        }

        for(int y=0;y<NELEM;y++){
            int Arow=globalrow+y*TILE;
            int Acol=i*TILE+localcol;
            
    
            if(Arow<M && Acol<K){
                sA[sizetile*y+localrow*TILE + localcol]=A[Arow*K + Acol];
            }
            else{
                sA[sizetile*y+localrow*TILE + localcol]=0.0f;
            }
        }
        
        __syncthreads();

        for(int k=0;k<TILE;k++){
            float b=sB[k*TILE+localcol ];
            for(int y=0;y<NELEM;y++){
                acc[y]+=sA[localrow*TILE+k+ y*sizetile]*b;
            }
        }

        __syncthreads();
    }

    for(int y=0;y<NELEM;y++){
        int grow=globalrow+y*TILE;
        if(grow<M && globalcol<N){
            C[grow*N+globalcol]=acc[y];
        }
    }
}

using namespace std;

int main(int argc,char *argv[]){
    if (argc < 4) {
        printf("Usage: ./matmul_cublas M K N\n");
        return 1;
    }

    int M = atoi(argv[1]);
    int K = atoi(argv[2]);
    int N = atoi(argv[3]);

    dim3 threads(32,32);
    int tx=(N+TILE-1)/TILE;
    int ty=(M+NELEM*TILE-1)/(NELEM*TILE);
    dim3 blocks(tx,ty);

    size_t sizeA=M*K*sizeof(float);
    size_t sizeB=N*K*sizeof(float);
    size_t sizeC=M*N*sizeof(float);
    float *A=(float*)malloc(sizeA);
    float *B=(float*)malloc(sizeB);
    float *C=(float*)malloc(sizeC);

    for(int i=0;i<M*K;i++){
        A[i]=1.0f;
    }
    for(int i=0;i<K*N;i++){
        B[i]=1.0f;
    }
    float *dA,*dB,*dC;
    cudaMalloc(&dA,sizeA);
    cudaMalloc(&dB,sizeB);
    cudaMalloc(&dC,sizeC);

    cudaEvent_t events[4];
    for(int i=0;i<4;i++) cudaEventCreate(&events[i]);

    cudaEventRecord(events[0]);
    cudaMemcpy(dA,A,sizeA,cudaMemcpyHostToDevice);
    cudaMemcpy(dB,B,sizeB,cudaMemcpyHostToDevice);

    cudaEventRecord(events[1]);
    matmul<<<blocks,threads>>>(dA,dB,dC,M,K,N);
    cudaEventRecord(events[2]);
    cudaDeviceSynchronize();

    cudaMemcpy(C,dC,sizeC,cudaMemcpyDeviceToHost);
    cudaEventRecord(events[3]);

    cudaEventSynchronize(events[2]);
    float mt=0.0f;
    cudaEventElapsedTime(&mt,events[1],events[2]);
    printf("kernel time: %.3f ms\n", mt);

    cudaEventSynchronize(events[3]);
    float mst=0.0f;
    cudaEventElapsedTime(&mst,events[0],events[3]);
    printf("overall time: %.3f ms\n", mst);

    for(int i=0;i<M*N;i++){
        if(fabs(C[i]-K)>1e-5){
            cout<<"wrong";
            free(A);
            free(B);
            free(C);
            cudaFree(dA);
            cudaFree(dB);
            cudaFree(dC);
            exit(1);
        }
    }

    cout<<"Works Correctly";
    free(A);
    free(B);
    free(C);
    cudaFree(dA);
    cudaFree(dB);
    cudaFree(dC);
    return 0;
}

Writing tiling3.cu


In [17]:
!nvcc tiling3.cu -o tiling3

In [19]:
!nvprof ./tiling3 1024 1025 1025

==300== NVPROF is profiling process 300, command: ./tiling3 1024 1025 1025
kernel time: 3.092 ms
overall time: 8.704 ms
Works Correctly==300== Profiling application: ./tiling3 1024 1025 1025
==300== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   40.54%  2.9225ms         1  2.9225ms  2.9225ms  2.9225ms  matmul(float*, float*, float*, int, int, int)
                   37.94%  2.7350ms         1  2.7350ms  2.7350ms  2.7350ms  [CUDA memcpy DtoH]
                   21.52%  1.5518ms         2  775.90us  771.19us  780.60us  [CUDA memcpy HtoD]
      API calls:   92.63%  181.85ms         3  60.618ms  94.926us  181.65ms  cudaMalloc
                    2.81%  5.5156ms         3  1.8385ms  927.53us  3.5856ms  cudaMemcpy
                    2.53%  4.9610ms       228  21.758us     108ns  1.5109ms  cuDeviceGetAttribute
                    1.49%  2.9235ms         1  2.9235ms  2.9235ms  2.9235ms  cudaDeviceSynchronize
             

In [34]:
%%writefile tiling4.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <math.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <iostream>

#define TILE 32
#define NELEM 4

__global__ void matmul(float *A,float *B,float *C,int M,int K,int N){
    __shared__ float sA[TILE*TILE*NELEM];
    __shared__ float sB[TILE*TILE*NELEM];
    
    int globalrow=threadIdx.y+blockIdx.y*blockDim.y*NELEM;
    int globalcol=threadIdx.x+blockIdx.x*blockDim.x*NELEM;
    int localrow=threadIdx.y;
    int localcol=threadIdx.x;

    float acc[NELEM][NELEM]={0};
    int  numtiles=(K+TILE-1)/TILE;
    int sizetile=TILE*TILE;
    
    for(int i=0;i<numtiles;i++){
         for(int y=0;y<NELEM;y++){
            int Brow=i*TILE+localrow;
            int Bcol=globalcol+y*TILE;
            if(Brow<K && Bcol<N){
                sB[sizetile*y+localrow*TILE + localcol]=B[Brow*N + Bcol];
            }
            else{
                sB[sizetile*y+localrow*TILE + localcol]=0.0f;
            }
        }

        for(int y=0;y<NELEM;y++){
            int Arow=globalrow+y*TILE;
            int Acol=i*TILE+localcol;
            
    
            if(Arow<M && Acol<K){
                sA[sizetile*y+localrow*TILE + localcol]=A[Arow*K + Acol];
            }
            else{
                sA[sizetile*y+localrow*TILE + localcol]=0.0f;
            }
        }
        
        __syncthreads();

        for(int k=0;k<TILE;k++){
            float a_frag[NELEM];
            float b_frag[NELEM];

            #pragma unroll
            for(int i=0;i<NELEM;i++)
                a_frag[i] =
                    sA[i*sizetile + localrow*TILE + k];

            #pragma unroll
            for(int j=0;j<NELEM;j++)
                b_frag[j] =
                    sB[j*sizetile + k*TILE + localcol];

            #pragma unroll
            for(int i=0;i<NELEM;i++)
                #pragma unroll
                for(int j=0;j<NELEM;j++)
                    acc[i][j] += a_frag[i] * b_frag[j];
        }

        __syncthreads();
    }

    #pragma unroll
    for(int i=0;i<NELEM;i++){

        int grow = globalrow + i*TILE;

        if(grow < M){
            #pragma unroll
            for(int j=0;j<NELEM;j++){

                int gcol = globalcol + j*TILE;

                if(gcol < N)
                    C[grow*N + gcol] = acc[i][j];
            }
        }
    }
}

using namespace std;

int main(int argc,char *argv[]){
    if (argc < 4) {
        printf("Usage: ./matmul_cublas M K N\n");
        return 1;
    }

    int M = atoi(argv[1]);
    int K = atoi(argv[2]);
    int N = atoi(argv[3]);

    dim3 threads(32,32);
    int tx=(N+NELEM*TILE-1)/(NELEM*TILE);
    int ty=(M+NELEM*TILE-1)/(NELEM*TILE);
    dim3 blocks(tx,ty);

    size_t sizeA=M*K*sizeof(float);
    size_t sizeB=N*K*sizeof(float);
    size_t sizeC=M*N*sizeof(float);
    float *A=(float*)malloc(sizeA);
    float *B=(float*)malloc(sizeB);
    float *C=(float*)malloc(sizeC);

    for(int i=0;i<M*K;i++){
        A[i]=1.0f;
    }
    for(int i=0;i<K*N;i++){
        B[i]=1.0f;
    }
    float *dA,*dB,*dC;
    cudaMalloc(&dA,sizeA);
    cudaMalloc(&dB,sizeB);
    cudaMalloc(&dC,sizeC);

    cudaEvent_t events[4];
    for(int i=0;i<4;i++) cudaEventCreate(&events[i]);

    cudaEventRecord(events[0]);
    cudaMemcpy(dA,A,sizeA,cudaMemcpyHostToDevice);
    cudaMemcpy(dB,B,sizeB,cudaMemcpyHostToDevice);

    cudaEventRecord(events[1]);
    matmul<<<blocks,threads>>>(dA,dB,dC,M,K,N);
    cudaEventRecord(events[2]);
    cudaDeviceSynchronize();

    cudaMemcpy(C,dC,sizeC,cudaMemcpyDeviceToHost);
    cudaEventRecord(events[3]);

    cudaEventSynchronize(events[2]);
    float mt=0.0f;
    cudaEventElapsedTime(&mt,events[1],events[2]);
    printf("kernel time: %.3f ms\n", mt);

    cudaEventSynchronize(events[3]);
    float mst=0.0f;
    cudaEventElapsedTime(&mst,events[0],events[3]);
    printf("overall time: %.3f ms\n", mst);

    for(int i=0;i<M*N;i++){
        if(fabs(C[i]-K)>1e-5){
            cout<<"wrong";
            free(A);
            free(B);
            free(C);
            cudaFree(dA);
            cudaFree(dB);
            cudaFree(dC);
            exit(1);
        }
    }

    cout<<"Works Correctly";
    free(A);
    free(B);
    free(C);
    cudaFree(dA);
    cudaFree(dB);
    cudaFree(dC);
    return 0;
}

Overwriting tiling4.cu


In [35]:
!nvcc tiling4.cu -o tiling4

In [37]:
!nvprof ./tiling4 1024 1025 1025

==489== NVPROF is profiling process 489, command: ./tiling4 1024 1025 1025
kernel time: 1.768 ms
overall time: 7.346 ms
Works Correctly==489== Profiling application: ./tiling4 1024 1025 1025
==489== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   45.37%  2.6489ms         1  2.6489ms  2.6489ms  2.6489ms  [CUDA memcpy DtoH]
                   27.32%  1.5949ms         2  797.45us  774.27us  820.63us  [CUDA memcpy HtoD]
                   27.31%  1.5941ms         1  1.5941ms  1.5941ms  1.5941ms  matmul(float*, float*, float*, int, int, int)
      API calls:   93.25%  180.40ms         3  60.134ms  96.559us  180.20ms  cudaMalloc
                    2.83%  5.4809ms         3  1.8270ms  985.95us  3.5067ms  cudaMemcpy
                    2.54%  4.9219ms       228  21.587us     105ns  1.4360ms  cuDeviceGetAttribute
                    0.82%  1.5941ms         1  1.5941ms  1.5941ms  1.5941ms  cudaDeviceSynchronize
             

In [38]:
%%writefile MLP.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <math.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <iostream>

#define TILE 32
#define NUMELEM 4

__global__ void matmul(float *A,float *B,float *C,int M,int K,int N){
    int localrow = threadIdx.y;
    int localcol = threadIdx.x;

    int baseRow = blockIdx.y * TILE * NUMELEM;
    int baseCol = blockIdx.x * TILE * NUMELEM;

    int globalrow = baseRow + localrow;
    int globalcol = baseCol + localcol;

    __shared__ float sA[TILE*TILE*NUMELEM];
    __shared__ float sB[TILE*TILE*NUMELEM];

    float acc[NUMELEM][NUMELEM] = {0};

    int sizetile = TILE*TILE;

    int numtiles = (K + TILE - 1) / TILE;

    for(int t=0; t<numtiles; t++){

        /* ---- Load A tiles (NUMELEM vertical tiles) ---- */

        for(int i=0;i<NUMELEM;i++){

            int Arow = globalrow + i*TILE;
            int Acol = t*TILE + localcol;

            if(Arow < M && Acol < K)
                sA[i*sizetile + localrow*TILE + localcol] =
                    A[Arow*K + Acol];
            else
                sA[i*sizetile + localrow*TILE + localcol] = 0.0f;
        }

        /* ---- Load B tiles (NUMELEM horizontal tiles) ---- */

        for(int j=0;j<NUMELEM;j++){

            int Brow = t*TILE + localrow;
            int Bcol = globalcol + j*TILE;

            if(Brow < K && Bcol < N)
                sB[j*sizetile + localrow*TILE + localcol] =
                    B[Brow*N + Bcol];
            else
                sB[j*sizetile + localrow*TILE + localcol] = 0.0f;
        }

        __syncthreads();

        /* ---- Compute ---- */

        #pragma unroll
        for(int k=0;k<TILE;k++){

            float a_frag[NUMELEM];
            float b_frag[NUMELEM];

            #pragma unroll
            for(int i=0;i<NUMELEM;i++)
                a_frag[i] =
                    sA[i*sizetile + localrow*TILE + k];

            #pragma unroll
            for(int j=0;j<NUMELEM;j++)
                b_frag[j] =
                    sB[j*sizetile + k*TILE + localcol];

            #pragma unroll
            for(int i=0;i<NUMELEM;i++)
                #pragma unroll
                for(int j=0;j<NUMELEM;j++)
                    acc[i][j] += a_frag[i] * b_frag[j];
        }

        __syncthreads();
    }

    /* ---- Write results ---- */
    #pragma unroll
    for(int i=0;i<NUMELEM;i++){

        int grow = globalrow + i*TILE;

        if(grow < M){
            #pragma unroll
            for(int j=0;j<NUMELEM;j++){

                int gcol = globalcol + j*TILE;

                if(gcol < N)
                    C[grow*N + gcol] = acc[i][j];
            }
        }
    }
}


__global__ void relu(float *A,int M,int N){
    int col=threadIdx.x+blockIdx.x*blockDim.x;
    int row=threadIdx.y+blockIdx.y*blockDim.y;

    if(col<N && row<M){
        int t=row*N+col;
        A[t]=fmaxf(0.0f,A[t]);
    }
}

using namespace std;

int main(int argc,char* argv[]){
    if (argc < 2) {
        printf("Usage: ./matmul_naive N\n");
        return 1;
    }

    int N = atoi(argv[1]);
    int B=32;
    size_t input_size=B*N*sizeof(float);
    size_t size=N*N*sizeof(float);
    float *W1=(float *)malloc(size);
    float *W2=(float *)malloc(size);
    float *X=(float *)malloc(input_size);
    float *Output=(float *)malloc(input_size);

    srand(time(nullptr));
    int lower=0,upper=50;
    for (int i = 0; i < N*N; i++) {
        W1[i] = lower + rand() % (upper - lower + 1);
        W2[i] = lower + rand() % (upper - lower + 1);
    }

    for(int i=0; i< B*N; i++){
        X[i]= lower + rand() % (upper - lower + 1);
    }

    float *d_W1,*d_W2,*d_X,*d_O1,*d_O2;
    cudaMalloc(&d_W1,size);
    cudaMalloc(&d_W2,size);
    cudaMalloc(&d_X,input_size);
    cudaMalloc(&d_O1,input_size);
    cudaMalloc(&d_O2,input_size);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    
    dim3 threadsPerBlock(32,32);
    int t=(N+TILE-1)/TILE;
    int tb=(B+TILE-1)/TILE;
    dim3 blocks(t,tb);

    int tt=(N+NUMELEM*TILE-1)/(NUMELEM*TILE);
    int ttb=(B+NUMELEM*TILE-1)/(NUMELEM*TILE);
    dim3 blockst(tt,ttb);

    cudaMemcpy(d_W1,W1,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_W2,W2,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_X,X,input_size,cudaMemcpyHostToDevice);

    cudaEventRecord(start);
    matmul<<<blockst,threadsPerBlock>>>(d_X,d_W1,d_O1,B,N,N);
    relu<<<blocks,threadsPerBlock>>>(d_O1,B,N);
    matmul<<<blockst,threadsPerBlock>>>(d_O1,d_W2,d_O2,B,N,N);
    cudaEventRecord(stop);
    cudaDeviceSynchronize();
    
    cudaEventSynchronize(stop);
    float timeEl=0;
    cudaEventElapsedTime(&timeEl,start,stop);
    printf("Total execution time on GPU: %.3f ms\n", timeEl);

    cudaMemcpy(Output,d_O2,input_size,cudaMemcpyDeviceToHost);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaError_t err = cudaGetLastError();

    // ----- CPU verification -----

    float *Y_cpu = (float*)malloc(input_size);
    float *Z_cpu = (float*)malloc(input_size);
    
    // Y = ReLU(W1 * X)
    for(int b=0;b<B;b++){
        for(int j=0;j<N;j++){
            double sum=0;
            for(int k=0;k<N;k++){
                sum += (double)X[b*N+k] * (double)W1[k*N+j];
            }
            Y_cpu[b*N+j] = fmaxf(0.0f,sum);
        }
    }
    
    // Z = W2 * Y
    for(int b=0;b<B;b++){
        for(int j=0;j<N;j++){
            double sum=0;
            for(int k=0;k<N;k++){
                sum += (double)Y_cpu[b*N+k] * (double)W2[k*N+j];
            }
            Z_cpu[b*N+j] = sum;
        }
    }
    
    // Compare
    for(int i=0;i<B*N;i++){ 
        float diff = fabs(Z_cpu[i] - Output[i]);
        float denom = fmaxf(fabs(Z_cpu[i]), 1.0f);
        
        if(diff / denom > 1e-4){
            printf("Mismatch at %d CPU=%f GPU=%f\n",i,Z_cpu[i],Output[i]);
            exit(1);
        }
    }
    
    printf("Verification Passed\n");
    
    free(Y_cpu);
    free(Z_cpu);

    if(err != cudaSuccess)
        printf("CUDA error: %s\n", cudaGetErrorString(err));
    else cout<<"Works Correctly"<<endl;
    cudaFree(d_W1);
    cudaFree(d_W2);
    cudaFree(d_X);
    cudaFree(d_O1);
    cudaFree(d_O2);
    free(W1);
    free(W2);
    free(X);
    free(Output);
    return 0;
}

Writing MLP.cu


In [39]:
!nvcc MLP.cu -o MLP

In [40]:
!nvprof ./MLP 1024

==533== NVPROF is profiling process 533, command: ./MLP 1024
Total execution time on GPU: 66.614 ms
Verification Passed
Works Correctly
==533== Profiling application: ./MLP 1024
==533== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   57.51%  1.5976ms         3  532.53us  13.088us  804.12us  [CUDA memcpy HtoD]
                   41.94%  1.1652ms         2  582.59us  582.39us  582.78us  matmul(float*, float*, float*, int, int, int)
                    0.44%  12.128us         1  12.128us  12.128us  12.128us  [CUDA memcpy DtoH]
                    0.12%  3.2640us         1  3.2640us  3.2640us  3.2640us  relu(float*, int, int)
      API calls:   70.93%  181.94ms         5  36.388ms  2.1200us  181.74ms  cudaMalloc
                   25.52%  65.453ms         3  21.818ms  4.9870us  65.440ms  cudaLaunchKernel
                    1.93%  4.9425ms       228  21.677us     104ns  1.4742ms  cuDeviceGetAttribute
                   

In [61]:
%%writefile MLP.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda.h>
#include <math.h>
#include <cstdlib>
#include <cuda_runtime.h>
#include <iostream>
#include <ctime>

#define TILE 32
#define NUMELEM 4

__global__ void matmul(float *A,float *B,float *C,int M,int K,int N){
    int localrow = threadIdx.y;
    int localcol = threadIdx.x;

    int baseRow = blockIdx.y * TILE * NUMELEM;
    int baseCol = blockIdx.x * TILE * NUMELEM;

    int globalrow = baseRow + localrow;
    int globalcol = baseCol + localcol;

    __shared__ float sA[TILE*TILE*NUMELEM];
    __shared__ float sB[TILE*TILE*NUMELEM];

    float acc[NUMELEM][NUMELEM] = {0};

    int sizetile = TILE*TILE;

    int numtiles = (K + TILE - 1) / TILE;

    for(int t=0; t<numtiles; t++){

        /* ---- Load A tiles (NUMELEM vertical tiles) ---- */

        for(int i=0;i<NUMELEM;i++){

            int Arow = globalrow + i*TILE;
            int Acol = t*TILE + localcol;

            if(Arow < M && Acol < K)
                sA[i*sizetile + localrow*TILE + localcol] =
                    A[Arow*K + Acol];
            else
                sA[i*sizetile + localrow*TILE + localcol] = 0.0f;
        }

        /* ---- Load B tiles (NUMELEM horizontal tiles) ---- */

        for(int j=0;j<NUMELEM;j++){

            int Brow = t*TILE + localrow;
            int Bcol = globalcol + j*TILE;

            if(Brow < K && Bcol < N)
                sB[j*sizetile + localrow*TILE + localcol] =
                    B[Brow*N + Bcol];
            else
                sB[j*sizetile + localrow*TILE + localcol] = 0.0f;
        }

        __syncthreads();

        /* ---- Compute ---- */

        #pragma unroll
        for(int k=0;k<TILE;k++){

            float a_frag[NUMELEM];
            float b_frag[NUMELEM];

            #pragma unroll
            for(int i=0;i<NUMELEM;i++)
                a_frag[i] =
                    sA[i*sizetile + localrow*TILE + k];

            #pragma unroll
            for(int j=0;j<NUMELEM;j++)
                b_frag[j] =
                    sB[j*sizetile + k*TILE + localcol];

            #pragma unroll
            for(int i=0;i<NUMELEM;i++)
                #pragma unroll
                for(int j=0;j<NUMELEM;j++)
                    acc[i][j] += a_frag[i] * b_frag[j];
        }

        __syncthreads();
    }

    /* ---- Write results ---- */
    #pragma unroll
    for(int i=0;i<NUMELEM;i++){

        int grow = globalrow + i*TILE;

        if(grow < M){
            #pragma unroll
            for(int j=0;j<NUMELEM;j++){

                int gcol = globalcol + j*TILE;

                if(gcol < N)
                    C[grow*N + gcol] = acc[i][j];
            }
        }
    }
}


__global__ void relu(float *A,int M,int N){
    int col=threadIdx.x+blockIdx.x*blockDim.x;
    int row=threadIdx.y+blockIdx.y*blockDim.y;

    if(col<N && row<M){
        int t=row*N+col;
        A[t]=fmaxf(0.0f,A[t]);
    }
}

using namespace std;

int main(int argc,char* argv[]){
    if (argc < 2) {
        printf("Usage: ./matmul_naive N\n");
        return 1;
    }

    int N = atoi(argv[1]);
    int B=32;
    size_t input_size=B*N*sizeof(float);
    size_t size=N*N*sizeof(float);
    float *W1=(float *)malloc(size);
    float *W2=(float *)malloc(size);
    float *X,*Output;
    cudaMallocHost(&X,input_size);
    cudaMallocHost(&Output,input_size);

    int chunk=B/4;

    srand(time(nullptr));
    int lower=0,upper=50;
    for (int i = 0; i < N*N; i++) {
        W1[i] = lower + rand() % (upper - lower + 1);
        W2[i] = lower + rand() % (upper - lower + 1);
    }

    for(int i=0; i< B*N; i++){
        X[i]= lower + rand() % (upper - lower + 1);
    }

    float *d_W1,*d_W2,*d_X,*d_O1,*d_O2;
    cudaMalloc(&d_W1,size);
    cudaMalloc(&d_W2,size);
    cudaMalloc(&d_X,input_size);
    cudaMalloc(&d_O1,input_size);
    cudaMalloc(&d_O2,input_size);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    
    dim3 threadsPerBlock(32,32);
    int t=(N+TILE-1)/TILE;
    int tb=(chunk+TILE-1)/TILE;
    dim3 blocks(t,tb);

    int tt=(N+NUMELEM*TILE-1)/(NUMELEM*TILE);
    int ttb=(chunk+NUMELEM*TILE-1)/(NUMELEM*TILE);
    dim3 blockst(tt,ttb);

    cudaMemcpy(d_W1,W1,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_W2,W2,size,cudaMemcpyHostToDevice);
    // cudaMemcpy(d_X,X,input_size,cudaMemcpyHostToDevice);
    cudaStream_t streams[4]; 
    for(int i=0;i<4;i++) cudaStreamCreate(&streams[i]);
    
    cudaEventRecord(start);

    for(int i=0;i<4;i++){
        int offset=chunk*i*N;
        float *x_chunk=d_X+offset;
        float *X_chunk=X+offset;
        float *O1_chunk=d_O1+offset;
        float *O_chunk=Output+offset;
        float *O2_chunk=d_O2+offset;
        
        cudaMemcpyAsync(x_chunk,X_chunk,chunk*N*sizeof(float),cudaMemcpyHostToDevice,streams[i]);
        matmul<<<blockst,threadsPerBlock,0,streams[i]>>>(x_chunk,d_W1,O1_chunk,chunk,N,N);
        relu<<<blocks,threadsPerBlock,0,streams[i]>>>(O1_chunk,chunk,N);
        matmul<<<blockst,threadsPerBlock,0,streams[i]>>>(O1_chunk,d_W2,O2_chunk,chunk,N,N);
        cudaMemcpyAsync(O_chunk,O2_chunk,chunk*N*sizeof(float),cudaMemcpyDeviceToHost,streams[i]);

    }
   
    cudaDeviceSynchronize();
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float timeEl=0;
    cudaEventElapsedTime(&timeEl,start,stop);
    printf("Total execution time on GPU: %.3f ms\n", timeEl);

    // cudaMemcpy(Output,d_O2,input_size,cudaMemcpyDeviceToHost);
    for(int i=0;i<4;i++) cudaStreamDestroy(streams[i]);
    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaError_t err = cudaGetLastError();

    // ----- CPU verification -----

    float *Y_cpu = (float*)malloc(input_size);
    float *Z_cpu = (float*)malloc(input_size);
    
    // Y = ReLU(W1 * X)
    for(int b=0;b<B;b++){
        for(int j=0;j<N;j++){
            double sum=0;
            for(int k=0;k<N;k++){
                sum += (double)X[b*N+k] * (double)W1[k*N+j];
            }
            Y_cpu[b*N+j] = fmaxf(0.0f,sum);
        }
    }
    
    // Z = W2 * Y
    for(int b=0;b<B;b++){
        for(int j=0;j<N;j++){
            double sum=0;
            for(int k=0;k<N;k++){
                sum += (double)Y_cpu[b*N+k] * (double)W2[k*N+j];
            }
            Z_cpu[b*N+j] = sum;
        }
    }
    
    // Compare
    for(int i=0;i<B*N;i++){ 
        float diff = fabs(Z_cpu[i] - Output[i]);
        float denom = fmaxf(fabs(Z_cpu[i]), 1.0f);
        
        if(diff / denom > 1e-4){
            printf("Mismatch at %d CPU=%f GPU=%f\n",i,Z_cpu[i],Output[i]);
            exit(1);
        }
    }
    
    printf("Verification Passed\n");
    
    free(Y_cpu);
    free(Z_cpu);

    if(err != cudaSuccess)
        printf("CUDA error: %s\n", cudaGetErrorString(err));
    else cout<<"Works Correctly"<<endl;
    cudaFree(d_W1);
    cudaFree(d_W2);
    cudaFree(d_X);
    cudaFree(d_O1);
    cudaFree(d_O2);
    free(W1);
    free(W2);
    cudaFreeHost(X);
    cudaFreeHost(Output);
    return 0;
}

Overwriting MLP.cu


In [62]:
!nvcc MLP.cu -o MLP

In [63]:
!nvprof ./MLP 1024

==1490== NVPROF is profiling process 1490, command: ./MLP 1024
Total execution time on GPU: 1.699 ms
Verification Passed
Works Correctly
==1490== Profiling application: ./MLP 1024
==1490== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   74.90%  4.6432ms         8  580.39us  576.09us  587.32us  matmul(float*, float*, float*, int, int, int)
                   24.57%  1.5230ms         6  253.83us  5.2800us  754.07us  [CUDA memcpy HtoD]
                    0.27%  16.671us         4  4.1670us  3.8720us  4.4790us  [CUDA memcpy DtoH]
                    0.26%  16.416us         4  4.1040us  4.0320us  4.1920us  relu(float*, int, int)
      API calls:   94.81%  184.88ms         2  92.442ms  5.4770us  184.88ms  cudaHostAlloc
                    2.45%  4.7774ms       228  20.953us     107ns  1.2974ms  cuDeviceGetAttribute
                    0.99%  1.9384ms         2  969.18us  943.30us  995.06us  cudaMemcpy
                   